## tl;dr

Les six fits R ont utilisé exactement les mêmes lignes effectives que les runs Python épinglés. L’écart absolu médian entre moteurs est de **6.08 points** et le maximum de **17.99 points**. Les résultats restent exploratoires : les runs Python disponibles n’ont qu’une chaîne et 20 draws, et R `ei` fournit 99 simulations d’importance sans R-hat multi-chaînes.

## Context & Methods

### Key Assumptions

- Panel nominal fixe de 3 000 communes (`panel_3000_seed_20260802`).
- H0A et H1, sur les législatives 1962, 1986 et 2022.
- Filtre de tomographie PyEI appliqué une seule fois avant les deux moteurs.
- Même ordre communal et même graine numérique 20260802.
- Famille King à normale tronquée, avec moteurs et priors non strictement identiques.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

HERE = Path.cwd()
if not (HERE / 'results').exists():
    HERE = HERE / 'analysis' / 'king_r_python_comparison'
RESULTS = HERE / 'results'
manifest = pd.read_csv(RESULTS / 'comparison_manifest.csv')
input_audit = pd.read_csv(RESULTS / 'input_audit.csv')
comparison = pd.read_csv(RESULTS / 'comparison_summary.csv')
diagnostics = pd.read_csv(RESULTS / 'diagnostic_comparison.csv')
highlights = json.loads((RESULTS / 'comparison_highlights.json').read_text(encoding='utf-8'))

## Data

Les six entrées communes et leur audit de parité.

In [2]:
display(input_audit[['comparison_key','nominal_rows','prepared_rows','excluded_during_preparation','effective_rows','excluded_degenerate_rows','duplicate_unit_ids','python_unit_order_match','max_group_count_roundtrip_error','max_vote_count_roundtrip_error']])
assert input_audit['python_unit_order_match'].all()
assert (input_audit['duplicate_unit_ids'] == 0).all()

,comparison_key,nominal_rows,prepared_rows,excluded_during_preparation,effective_rows,excluded_degenerate_rows,duplicate_unit_ids,python_unit_order_match,max_group_count_roundtrip_error,max_vote_count_roundtrip_error
0,leg_1962_r1__H0A,3000,2985,15,2963,22,0,True,9.094947e-13,1.818989e-12
1,leg_1962_r1__H1,3000,2985,15,2925,60,0,True,3.637979e-12,9.094947e-13
2,leg_1986_r1__H0A,3000,2962,38,2928,34,0,True,7.275958e-12,9.094947e-13
3,leg_1986_r1__H1,3000,2962,38,2935,27,0,True,7.275958e-12,1.818989e-12
4,leg_2022_r1__H0A,3000,3000,0,2820,180,0,True,7.275958e-12,1.818989e-12
5,leg_2022_r1__H1,3000,3000,0,2813,187,0,True,1.818989e-12,1.818989e-12


## Results

Comparaison des estimations agrégées β1 et β2.

In [3]:
cols = ['fit_group_label','effective_n_python','estimate_python','lower_python','upper_python','estimate_r','lower_r','upper_r','difference_r_minus_python','absolute_difference_pp','interval_overlap_ratio']
display(comparison[cols].round(4))

,fit_group_label,effective_n_python,estimate_python,lower_python,upper_python,estimate_r,lower_r,upper_r,difference_r_minus_python,absolute_difference_pp,interval_overlap_ratio
0,H0A 1962 - beta1 cible,2963,0.2947,0.2946,0.2948,0.2667,0.2564,0.2765,-0.0280,2.8026,0.0000
1,H0A 1962 - beta2 autres,2963,0.4173,0.4170,0.4175,0.3909,0.3757,0.4067,-0.0264,2.6397,0.0000
2,H0A 1986 - beta1 cible,2928,0.1985,0.1985,0.1986,0.2295,0.2189,0.2395,0.0309,3.0934,0.0000
3,H0A 1986 - beta2 autres,2928,0.2593,0.2592,0.2593,0.1917,0.1785,0.2056,-0.0676,6.7553,0.0000
4,H0A 2022 - beta1 cible,2820,0.4979,0.4894,0.5082,0.5547,0.5468,0.5617,0.0568,5.6778,0.0000
5,H0A 2022 - beta2 autres,2820,0.5360,0.5220,0.5438,0.4711,0.4646,0.4784,-0.0649,6.4913,0.0000
6,H1 1962 - beta1 cible,2925,0.5356,0.4123,0.6230,0.4219,0.4183,0.4255,-0.1137,11.3717,0.0344
7,H1 1962 - beta2 autres,2925,0.2249,0.0693,0.4315,0.4048,0.3990,0.4105,0.1799,17.9867,0.0316
8,H1 1986 - beta1 cible,2935,0.4287,0.4049,0.4451,0.5105,0.4879,0.5350,0.0818,8.1809,0.0000
9,H1 1986 - beta2 autres,2935,0.4664,0.4427,0.4984,0.3698,0.3380,0.3991,-0.0966,9.6580,0.0000


In [4]:
summary = comparison.groupby(['scenario_id','estimand']).agg(
    median_abs_diff_pp=('absolute_difference_pp','median'),
    max_abs_diff_pp=('absolute_difference_pp','max'),
    mean_overlap=('interval_overlap_ratio','mean'),
).reset_index()
display(summary.round(3))

,scenario_id,estimand,median_abs_diff_pp,max_abs_diff_pp,mean_overlap
0,H0A,b_1,3.093,5.678,0.000
1,H0A,b_2,6.491,6.755,0.000
2,H1,b_1,8.181,11.372,0.011
3,H1,b_2,9.658,17.987,0.011


### Diagnostics

Les diagnostics ne sont pas directement homologues entre NUTS/PyMC et l’importance sampling de R `ei`.

In [5]:
diag_cols = ['comparison_key','engine','fit_status','diagnostic_status','diagnostic_reason','effective_n','elapsed_seconds','draws','chains','mcmc_divergences','mcmc_min_ess_bulk','importance_simulations','hessian_positive_definite','max_accounting_reconstruction_error']
display(diagnostics.reindex(columns=diag_cols).round(5))

,comparison_key,engine,fit_status,diagnostic_status,diagnostic_reason,effective_n,elapsed_seconds,draws,chains,mcmc_divergences,mcmc_min_ess_bulk,importance_simulations,hessian_positive_definite,max_accounting_reconstruction_error
0,leg_1962_r1__H0A,Python PyEI 1.1.4,success,fail,Pinned exploratory run: one chain and 20 retai...,2963,40.61047,20.0,1.0,14.0,2.04283,NaN,NaN,NaN
1,leg_1962_r1__H0A,R ei 1.3-3,success,caveat,Fit and accounting checks pass; R ei uses 99 i...,2963,123.53000,NaN,NaN,NaN,NaN,99.0,True,0.0
2,leg_1962_r1__H1,Python PyEI 1.1.4,success,fail,Pinned exploratory run: one chain and 20 retai...,2925,63.82807,20.0,1.0,0.0,2.04220,NaN,NaN,NaN
3,leg_1962_r1__H1,R ei 1.3-3,success,caveat,Fit and accounting checks pass; R ei uses 99 i...,2925,108.79000,NaN,NaN,NaN,NaN,99.0,True,0.0
4,leg_1986_r1__H0A,Python PyEI 1.1.4,success,fail,Pinned exploratory run: one chain and 20 retai...,2928,28.54779,20.0,1.0,20.0,2.03678,NaN,NaN,NaN
5,leg_1986_r1__H0A,R ei 1.3-3,success,caveat,Fit and accounting checks pass; R ei uses 99 i...,2928,113.04000,NaN,NaN,NaN,NaN,99.0,True,0.0
6,leg_1986_r1__H1,Python PyEI 1.1.4,success,fail,Pinned exploratory run: one chain and 20 retai...,2935,95.30917,20.0,1.0,0.0,2.08619,NaN,NaN,NaN
7,leg_1986_r1__H1,R ei 1.3-3,success,caveat,Fit and accounting checks pass; R ei uses 99 i...,2935,114.97000,NaN,NaN,NaN,NaN,99.0,True,0.0
8,leg_2022_r1__H0A,Python PyEI 1.1.4,success,fail,Pinned exploratory run: one chain and 20 retai...,2820,63.97553,20.0,1.0,0.0,2.07900,NaN,NaN,NaN
9,leg_2022_r1__H0A,R ei 1.3-3,success,caveat,Fit and accounting checks pass; R ei uses 99 i...,2820,108.07000,NaN,NaN,NaN,NaN,99.0,True,0.0


## Takeaways

1. La parité des inputs est vérifiée par effectif, ordre des identifiants, fermeture comptable et SHA-256.
2. Les écarts observés ne peuvent donc pas être attribués à un changement de composition des communes.
3. Ils peuvent en revanche provenir des priors, de l’optimisation/simulation et du faible budget de tirages du run Python existant.
4. Une conclusion substantielle demanderait un rerun Python multi-chaînes et, idéalement, une implémentation R du même modèle KRT `king99` bêta-binomial.